In [ ]:
# Colab bootstrap — auto-clone repo on Google Colab, no-op locally
import os, sys, subprocess

REPO = "https://github.com/jongmoonha/AI-PHM_Graduate.git"
DIR  = "AI-PHM_Graduate"

try:
    import google.colab  # type: ignore
    target = '/content/' + DIR
    if not os.path.isdir(target):
        subprocess.run(["git", "clone", REPO, target], check=True)
    os.chdir(target)
    print('Google Colab detected. Working directory:', os.getcwd())
except ImportError:
    print('Local environment detected. Working directory:', os.getcwd())


# 힐베르트 포락선(envelope) 분석: 반복되는 충격의 주기 찾기

빠르게 진동하는 반송파(carrier)의 진폭이 **느린 변조**(modulation)에 의해 변하는 신호에서는 원 신호의 FFT 만으로 변조의 주기를 확인하기 어렵다. 힐베르트 변환으로 얻은 **해석 신호의 크기**, 즉 포락선 $|x_a(t)| = \sqrt{x(t)^2 + \hat{x}(t)^2}$ 은 고주파 반송파를 제거하고 엔진 폭발의 반복, 충격의 반복, 베어링 결함의 임펄스와 같은 **느린 진폭 변화**만을 남긴다. 포락선을 다시 FFT 로 분석하면 시간 파형에서는 잘 보이지 않던 **충격의 반복 주파수**가 뚜렷하게 나타난다.

이 노트북의 목표는 다음과 같다.

1. **실습 1, 2.** 오토래시(autorash) 결함이 있을 때 발생하는 엔진음에서 크고 작은 펄스가 번갈아 반복되는 주기를 포락선 스펙트럼으로 확인한다.
2. **실습 3, 4.** 충격이 거의 같은 간격으로 반복되는 노킹 엔진음에서 포락선 스펙트럼에 충격의 간격이 어떻게 나타나는지 본다.
3. **실습 5 ~ 8.** 베어링 내륜 결함 신호(CWRU DE_IR)에 대역필터를 적용한 뒤 포락선을 구하여 **BPFI** 성분을 검출한다. 이 예는 실제 진단 상황에 해당한다.

## 공통 임포트와 데이터 로드

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import utils

try:
    import librosa
except ImportError:
    %pip install librosa
    import librosa

In [ ]:
# autorash 결함 시 엔진음 / Knocking 엔진음
v_a, fs_a = librosa.load('./data/autorash.mp3', sr=None)
v_k, fs_k = librosa.load('./data/knocking.mp3',  sr=None)

# 두 신호 길이 맞춤 (autorash 를 Knocking 길이에 맞춰 잘라줌)
slice_length = len(v_k)
v_a = v_a[:slice_length]

t_a = np.arange(len(v_a)) / fs_a
t_k = np.arange(len(v_k)) / fs_k

print('Engine sound with autorash fault:', v_a.shape, 'fs =', fs_a)
print('Knocking                        :', v_k.shape, 'fs =', fs_k)

## 실습 1. autorash 결함 시 엔진음의 시간 영역 포락선

이 엔진음은 수백 Hz ~ 수 kHz 의 빠른 진동을 반송파로 하여 **수십 Hz 의 반복 주파수**로 펄스가 반복되는 신호이다. 먼저 시간 영역에서 원 신호 위에 힐베르트 포락선을 겹쳐 그려, 포락선이 빠른 반송파의 위쪽 윤곽선을 어떻게 따라가는지 확인한다. 이어서 짧은 구간(0.60 ~ 0.72 s)을 확대하여 *개별 펄스* 하나하나가 포락선의 피크와 어떻게 대응하는지 본다.

In [ ]:
# Hilbert envelope of the raw waveform
v_a_env = utils.hilbert_envelope(v_a)

plt.figure(figsize=(8, 8))

plt.subplot(2, 1, 1)
plt.plot(t_a, v_a,      'C0', label='Engine sound with autorash fault')
plt.plot(t_a, v_a_env,  'C3', label='Envelope')
plt.xlabel('Time [sec]')
plt.ylabel('x(t)')
plt.title('Engine sound with autorash fault: time-domain waveform with envelope')
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(t_a, v_a,      'C0')
plt.plot(t_a, v_a_env,  'C3')
plt.xlim([0.60, 0.72])
plt.xlabel('Time [sec]')
plt.ylabel('x(t)')
plt.title('Engine sound with autorash fault: waveform with envelope (zoom 0.60 - 0.72 s)')

plt.subplots_adjust(hspace=0.35)
plt.show()

**관찰.** 포락선(Envelope)이 원 신호의 위쪽 윤곽선을 따라간다. 원 신호를 필터 없이 그대로 사용하였으므로 포락선도 짧은 간격으로 불규칙하게 변동하지만, 펄스가 지나갈 때마다 포락선이 커지는 것은 분명히 보인다. 전체 구간에서는 큰 펄스 12개가 약 95 ms 간격으로 뚜렷하고, 그 사이의 한가운데마다 작은 펄스가 포락선의 변동 범위보다 조금 높게 나타난다. 큰 펄스와 작은 펄스를 모두 세면 간격은 약 47 ms 이다. 확대 구간에는 0.62 s 와 0.71 s 의 큰 펄스와, 그 사이 0.67 s 부근의 작은 펄스가 포함된다. 다음 실습에서 이 윤곽선을 주파수 영역으로 옮기면 이웃한 펄스 사이의 간격(약 47 ms)이 펄스 반복 주파수(약 21 Hz)의 저주파 성분으로 나타난다.

---

## 실습 2. autorash 결함 시 엔진음의 원 스펙트럼과 포락선 스펙트럼 비교

같은 신호의 **원 스펙트럼**과 **포락선 스펙트럼**을 4분할로 비교한다. 위 두 패널에는 원 신호의 스펙트럼을 0 ~ 5000 Hz 와 0 ~ 100 Hz 의 두 범위로, 아래 두 패널에는 포락선의 스펙트럼을 같은 두 범위로 그린다. 원 스펙트럼에는 100 Hz 부근의 엔진 고조파 성분과 1 ~ 5 kHz 의 넓은 성분이 있지만 펄스 반복 주파수에는 성분이 없다. 반면 포락선 스펙트럼에는 0 ~ 100 Hz 영역에 펄스 반복 주파수와 그 고조파의 성분이 뚜렷하게 나타난다.

In [ ]:
# Raw waveform spectrum
F_raw_a, A_raw_a = utils.fft(v_a     - v_a.mean(),     fs_a)
# Envelope spectrum (별도 이름으로 시간영역 v_a_env 를 덮어쓰지 않도록 주의)
F_env_a, A_env_a = utils.fft(v_a_env - v_a_env.mean(), fs_a)

plt.figure(figsize=(10, 10))

plt.subplot(4, 1, 1)
plt.plot(F_raw_a, A_raw_a, 'C0')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Engine sound with autorash fault: single-sided FFT (0 - 5000 Hz)')
plt.xlim([0, 5000])

plt.subplot(4, 1, 2)
plt.plot(F_raw_a, A_raw_a, 'C0')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Engine sound with autorash fault: single-sided FFT (zoom 0 - 100 Hz)')
plt.xlim([0, 100])

plt.subplot(4, 1, 3)
plt.plot(F_env_a, A_env_a, 'C3')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Envelope of engine sound with autorash fault: single-sided FFT (0 - 5000 Hz)')
plt.xlim([0, 5000])

plt.subplot(4, 1, 4)
plt.plot(F_env_a, A_env_a, 'C3')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Envelope of engine sound with autorash fault: FFT (zoom 0 - 100 Hz)')
plt.xlim([0, 100])

plt.subplots_adjust(hspace=0.6)
plt.show()

**관찰.** 위 두 패널의 원 스펙트럼에서 가장 큰 성분들은 84, 127, 168 Hz 부근에 나타나고, 1 ~ 5 kHz 에는 성분이 넓게 퍼져 있다. 0 ~ 100 Hz 확대에서도 62 Hz 와 84 Hz 부근의 성분이 보일 뿐 펄스 반복 주파수(약 21 Hz)에는 성분이 없다. 반면 아래 두 패널의 포락선 스펙트럼에서는 0 ~ 100 Hz 확대 구간의 약 21 Hz 에 **가장 큰 성분**이 나타나고 그 정수배(약 42, 63 Hz)에도 성분이 발생하므로, 펄스 반복 주파수를 분명하게 확인할 수 있다. 10 Hz 와 31 Hz 부근의 성분은 큰 펄스와 작은 펄스가 번갈아 나타나는 약 94 ms 의 주기에 의한 것이다. 원 스펙트럼에서는 펄스 반복 주파수에 성분이 없지만, 포락선 스펙트럼은 복조를 통해 펄스의 **반복 주기**를 보여 준다.

---

## 실습 3. 노킹 엔진음의 시간 영역 포락선

노킹(knocking) 음원은 짧은 충격 펄스가 거의 같은 간격으로 반복되는 신호이다. 개별 펄스는 구조의 공진을 가진하므로 고주파 파형으로 관찰된다. 먼저 시간 영역에서 원 신호 위에 힐베르트 포락선을 겹쳐 그려, 포락선이 충격 펄스의 윤곽을 어떻게 따라가는지 본다.

In [ ]:
v_k_env = utils.hilbert_envelope(v_k)

plt.figure(figsize=(8, 8))

plt.subplot(2, 1, 1)
plt.plot(t_k, v_k,      'C0', label='Knocking')
plt.plot(t_k, v_k_env,  'C3', label='Envelope')
plt.xlabel('Time [sec]')
plt.ylabel('x(t)')
plt.title('Knocking time-domain waveform with envelope')
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(t_k, v_k,      'C0')
plt.plot(t_k, v_k_env,  'C3')
plt.xlim([0.60, 0.72])
plt.xlabel('Time [sec]')
plt.ylabel('x(t)')
plt.title('Knocking waveform with envelope (zoom 0.60 - 0.72 s)')

plt.subplots_adjust(hspace=0.35)
plt.show()

**관찰.** 포락선(Envelope)이 충격 펄스의 위쪽 윤곽선을 따라간다. 전체 구간에서 큰 충격은 약 77 ms 마다 거의 정확하게 반복되고, 큰 충격 사이에는 크기가 절반에 못 미치는 작은 충격이 대개 하나씩 있다. 확대 구간에서는 0.616 s 와 0.693 s 의 큰 충격 두 개와 0.66 s 부근의 작은 충격 하나를 볼 수 있다. 노킹 엔진음은 autorash 결함 시 엔진음보다 충격이 짧고 간격이 고르며, 다음 실습의 포락선 스펙트럼에서는 이 특징이 뚜렷한 고조파 성분으로 나타난다.

---

## 실습 4. 노킹 엔진음의 원 스펙트럼과 포락선 스펙트럼 비교

같은 4분할 비교를 노킹 신호에 적용한다. 큰 충격의 간격이 약 77 ms 로 고르므로, 포락선 스펙트럼의 저주파 영역에서는 그 역수(약 13 Hz)와 정수배의 주파수에 성분이 나타날 것으로 예상된다. 두 신호의 차이가 포락선 스펙트럼이라는 같은 도구에서 어떻게 다르게 보이는지 확인한다.

In [ ]:
F_raw_k, A_raw_k = utils.fft(v_k     - v_k.mean(),     fs_k)
F_env_k, A_env_k = utils.fft(v_k_env - v_k_env.mean(), fs_k)

plt.figure(figsize=(10, 10))

plt.subplot(4, 1, 1)
plt.plot(F_raw_k, A_raw_k, 'C0')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Knocking single-sided FFT (0 - 5000 Hz)')
plt.xlim([0, 5000])

plt.subplot(4, 1, 2)
plt.plot(F_raw_k, A_raw_k, 'C0')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Knocking single-sided FFT (zoom 0 - 100 Hz)')
plt.xlim([0, 100])

plt.subplot(4, 1, 3)
plt.plot(F_env_k, A_env_k, 'C3')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Envelope of Knocking single-sided FFT (0 - 5000 Hz)')
plt.xlim([0, 5000])

plt.subplot(4, 1, 4)
plt.plot(F_env_k, A_env_k, 'C3')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Envelope of Knocking FFT (zoom 0 - 100 Hz)')
plt.xlim([0, 100])

plt.subplots_adjust(hspace=0.6)
plt.show()

**관찰.** 위 두 패널의 노킹 원 스펙트럼에서는 0.3 ~ 2 kHz 와 3.5 ~ 5 kHz 에 에너지가 넓게 퍼져 있으며, 원 신호 에너지의 약 40 %는 이 패널의 범위 밖인 5 kHz 위에 있다. 0 ~ 100 Hz 확대에는 성분이 하나도 나타나지 않으므로 충격의 반복 구조를 확인할 수 없다. 아래 두 패널의 포락선 스펙트럼에서는 0 ~ 100 Hz 확대 구간의 약 13 Hz 와 그 정수배(26, 52, 65, 78 Hz 등)에 뚜렷한 성분이 일정한 간격으로 나타난다. 가장 큰 성분이 13 Hz 가 아니라 26 Hz 에 있는 것은, 한 주기(약 77 ms) 안에 큰 충격과 작은 충격이 하나씩 있어 충격이 대략 그 절반의 간격으로 발생하기 때문이다. 충격이 고른 간격으로 반복될수록 포락선 스펙트럼의 고조파 성분이 뚜렷해진다.

---

## 실습 5. CWRU 베어링의 결함 주파수(BPFI) 계산

베어링 내륜 결함은 볼이 결함부를 통과할 때마다 **짧은 임펄스**를 발생시키며, 이 임펄스는 베어링 하우징의 **고주파 공진**(수 kHz)을 가진한다. 즉 측정 신호는 *고주파 공진 반송파 × 저주파 결함 반복(BPFI) 변조*의 형태이다.

CWRU 벤치마크 데이터 `data/data_fault_DE_IR.csv` 를 불러와 기대되는 결함 주파수 BPFI 를 계산한다. 이 데이터는 Drive End(DE) 베어링의 내륜 결함 신호이며, 회전 속도는 약 1772 rpm, 샘플링 주파수는 $f_s = 12000$ Hz 이다. DE SKF6205 베어링의 BPFI 배수는 **5.4152**이므로 **BPFI ≈ 5.4152 × (1772/60) ≈ 160 Hz** 가 기대된다. 이 값은 이후 실습 7과 8의 포락선 스펙트럼에서 BPFI 가 검출되는지 확인하는 기준이 된다.

In [ ]:
data = np.array(pd.read_csv('./data/data_fault_DE_IR.csv'))
fs_f = 12000
v_f  = data[:, 1]
N_f  = len(v_f)
t_f  = np.arange(N_f) / fs_f

rpm       = 1772
f_shaft   = rpm / 60.0
BPFI      = 5.4152 * f_shaft   # DE SKF6205 inner-race
print(f'N = {N_f}, duration = {N_f/fs_f:.2f} s')
print(f'Shaft frequency : {f_shaft:.3f} Hz')
print(f'BPFI            : {BPFI:.3f} Hz')

**관찰.** 축 회전 주파수는 약 29.5 Hz(1772 / 60)이고 BPFI 는 약 160 Hz 이다. 이 두 값이 이후 실습의 기준이 된다. BPFI 성분은 실습 6의 원 스펙트럼에서는 두드러지지 않더라도, 실습 7에서 공진 대역에 대역필터를 적용하고 포락선을 구하면 실습 8의 포락선 스펙트럼에서 가장 크게 나타날 것이다.

---

## 실습 6. CWRU 베어링의 원 시간 파형과 원 스펙트럼

CWRU 결함 신호의 원 시간 파형은 0 ~ 0.3 s 를 확대하여, 원 스펙트럼은 0 ~ 6000 Hz 전체와 0 ~ 500 Hz 확대의 두 범위로 그린다. 0 ~ 500 Hz 확대에는 BPFI ≈ 160 Hz 의 위치를 파선으로 표시하여 원 스펙트럼에서 BPFI 성분이 두드러지는지 확인한다. 두드러지지 않는 것이 예상한 결과이며, 그 이유가 다음 실습에서 대역필터를 적용한 뒤 포락선을 구하는 동기가 된다.

In [ ]:
F_raw_f, A_raw_f = utils.fft(v_f - v_f.mean(), fs_f)

plt.figure(figsize=(10, 7))

plt.subplot(3, 1, 1)
plt.plot(t_f, v_f, 'C0')
plt.xlim([0.0, 0.3])
plt.xlabel('Time [sec]');  plt.ylabel('x(t)')
plt.title('Bearing inner-race fault raw waveform (zoom 0 - 0.3 s)')

plt.subplot(3, 1, 2)
plt.plot(F_raw_f, A_raw_f, 'C0')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Raw spectrum (0 - 6000 Hz)')
plt.xlim([0, 6000])

plt.subplot(3, 1, 3)
plt.plot(F_raw_f, A_raw_f, 'C0')
plt.axvline(BPFI, color='C3', linestyle='--', label=f'BPFI = {BPFI:.1f} Hz')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Raw spectrum (zoom 0 - 500 Hz)')
plt.xlim([0, 500])
plt.legend()

plt.subplots_adjust(hspace=0.6)
plt.show()

**관찰.** 확대한 시간 파형에서는 큰 충격 펄스가 곳곳에 나타나지만 그 간격이 일정하게 보이지 않으며, 0 ~ 6000 Hz 의 원 스펙트럼에서는 수 kHz 의 공진 대역이 지배적이다. 0 ~ 500 Hz 확대에서는 BPFI 파선의 위치에도 작은 성분이 있다. 그러나 같은 범위에 이보다 큰 성분이 여섯 개 있고, 가장 큰 약 360 Hz 성분은 BPFI 성분의 약 6배이므로 BPFI 위치의 성분만으로는 BPFI 를 식별할 수 없다. 이는 결함의 에너지가 **공진 대역에 포함되어 있어** 저주파에서는 두드러지지 않기 때문이다. 다음 실습에서는 이 공진 대역만 대역필터로 분리한 뒤 포락선으로 복조한다.

---

## 실습 7. CWRU 베어링의 공진 대역 대역필터와 힐베르트 포락선

CWRU DE 내륜 결함 신호의 주요 공진 대역인 **3500 ~ 5500 Hz** 를 `utils.filtering_zerophase` 로 추출한 뒤, 필터링한 신호에 `utils.hilbert_envelope` 를 적용하여 포락선을 구한다. 시간 영역에서 대역필터를 통과한 신호(Bandpass) 위에 포락선(Envelope)을 겹쳐 그리고, 포락선만 0 ~ 0.1 s 로 확대하여 결함 임펄스의 반복 구조가 눈으로 보이는지 확인한다.

In [ ]:
# Resonance band bandpass + envelope
v_f_bp  = utils.filtering_zerophase(v_f, fs_f, 'band', f_low=3500, f_high=5500, order=4)
v_f_env = utils.hilbert_envelope(v_f_bp)

plt.figure(figsize=(10, 6))

plt.subplot(2, 1, 1)
plt.plot(t_f, v_f_bp, 'C0', label='Bandpass 3500-5500 Hz')
plt.plot(t_f, v_f_env, 'C3', label='Envelope')
plt.xlim([0.0, 0.3])
plt.xlabel('Time [sec]');  plt.ylabel('x(t)')
plt.title('Bandpass waveform with envelope (zoom 0 - 0.3 s)')
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(t_f, v_f_env, 'C3')
plt.xlim([0.0, 0.1])
plt.xlabel('Time [sec]');  plt.ylabel('Envelope')
plt.title('Envelope only (zoom 0 - 0.1 s)')

plt.subplots_adjust(hspace=0.5)
plt.show()

**관찰.** 대역필터를 통과한 신호는 3500 ~ 5500 Hz 의 빠른 진동이며, 포락선은 그 반송파를 제거하고 진폭의 변화만 남긴다. 포락선에는 통과 대역폭(2 kHz)에 해당하는 빠른 변동이 있고, 그 위에 **충격에 의한 포락선 피크**가 나타난다. 포락선만 확대한 0 ~ 0.1 s 패널에서는 15, 27, 40, 84 ms 부근에 큰 충격이 보이고 그 사이에 작은 충격이 섞여 있다. 결함 주기는 약 6.25 ms(≈ 1/160 Hz)이지만, 충격의 세기가 축의 회전 각도에 따라 크게 달라지고 잡음도 포함되어 있으므로 충격 사이의 간격이 고르지 않다. 그래서 이 파형에서 결함 주기를 눈으로 측정하기는 어려우며, 다음 실습에서 이 포락선을 주파수 분석하여 BPFI 를 정량적으로 확인한다.

---

## 실습 8. CWRU 베어링의 포락선 스펙트럼과 BPFI 고조파 검출

포락선은 항상 양수여서 큰 DC 성분을 가지므로, 실습 7의 포락선에서 **평균을 제거한 뒤 FFT** 를 적용한다. 스펙트럼을 0 ~ 1000 Hz 와 0 ~ 500 Hz 의 두 범위로 그리고, 0 ~ 500 Hz 패널에는 BPFI 의 1, 2, 3배 위치를 파선(`axvline`)으로 표시하여 결함 주파수와 그 고조파의 성분이 일정한 간격으로 나타나는지 확인한다. 이것이 산업 현장에서 가장 널리 쓰이는 베어링 진단 파이프라인의 최종 그림이다.

In [ ]:
F_env_f, A_env_f = utils.fft(v_f_env - v_f_env.mean(), fs_f)

plt.figure(figsize=(10, 6))

plt.subplot(2, 1, 1)
plt.plot(F_env_f, A_env_f, 'C3')
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Envelope spectrum (0 - 1000 Hz)')
plt.xlim([0, 1000])

plt.subplot(2, 1, 2)
plt.plot(F_env_f, A_env_f, 'C3')
for k in range(1, 4):
    plt.axvline(k * BPFI, color='k', linestyle='--', alpha=0.6, zorder=0,  # 점선을 데이터 뒤에
                label=(f'k x BPFI' if k == 1 else None))
plt.xlabel('Frequency [Hz]');  plt.ylabel('|X(f)|')
plt.title('Envelope spectrum with BPFI harmonics (zoom 0 - 500 Hz)')
plt.xlim([0, 500])
plt.legend()

plt.subplots_adjust(hspace=0.5)
plt.show()

**관찰.** 포락선 스펙트럼에서는 **BPFI ≈ 160 Hz** 의 성분이 가장 크며, 진폭은 약 0.0175로 잡음 바닥의 약 43배이다. 2 × BPFI(≈ 320 Hz) 성분은 진폭이 약 0.003으로 바로 옆의 성분보다 조금 클 뿐이다. 3 × BPFI(≈ 480 Hz) 성분은 진폭이 약 0.0015로 잡음 바닥의 약 3.6배이지만 주변의 잡음 피크와 크기가 비슷하다. 따라서 이 신호에서 정수배의 성분 가운데 뚜렷한 것은 1배 성분뿐이다. 3 × BPFI 파선 왼쪽의 약 453 Hz 성분은 BPFI 의 정수배가 아니다. 160 Hz 아래의 30, 59, 89, 118 Hz 성분은 축 회전 주파수(약 29.5 Hz)의 정수배이다. 42, 71, 101, 130 Hz 성분은 BPFI 에서 축 회전 주파수의 1 ~ 4배만큼 떨어진 사이드밴드이며, 17 Hz 성분도 같은 사이드밴드에 속한다. 내륜은 축과 함께 회전하므로 결함은 한 회전마다 하중 구역을 통과한다. 그 결과 충격의 세기가 축 회전 주기로 변동하여 위의 성분들이 발생하며, 이는 내륜 결함을 뒷받침하는 증거이다. 다만 결함 주파수는 가장 큰 성분을 기준으로 읽지 않고, 이론으로 정해진 BPFI 의 정수배 위치에서 읽는다. 실습 1 ~ 4에서 autorash 결함 시 엔진음과 노킹 엔진음으로 확인한 "포락선은 반복되는 충격의 주기를 보여 준다"는 원리가 **회전기계 결함 진단의 표준 파이프라인**으로 그대로 이어진다.

---

## 정리

- 포락선은 **고주파 반송파를 제거하고 저주파 변조 성분만 남기는 복조 연산**이다.
- `utils.hilbert_envelope(x)` 한 줄이면 힐베르트 변환으로 얻은 해석 신호의 크기를 계산할 수 있다.
- 베어링 진단의 표준 절차는 **공진 대역 대역필터 → 포락선 → 주파수 분석**이며, 이 절차로 결함 특성 주파수(BPFI, BPFO 등)와 그 고조파를 확인한다.